# 08 — FINAL Statistical Tests: McNemar + DeLong + Holm–Bonferroni

**NO TRAINING IN THIS NOTEBOOK.**

This notebook uses the **locked final five-model benchmark** and performs:

1. **Pairwise McNemar tests** on the 3-class test-set correctness vectors.
2. **Pairwise DeLong tests** on clinical Cataract-v-Normal ROC AUC.
3. **Holm–Bonferroni correction** separately within the McNemar and DeLong test families.
4. A manuscript-ready pairwise comparison table.
5. A concise significance summary.

Five models → 10 pairwise comparisons per statistical-test family.

Locked test set:
- Total test images = 2,587
- Cataract = 854
- Normal = 977
- Not Eye = 756
- Clinical Cataract-v-Normal subset = 1,831

Significance level: α = 0.05.

In [ ]:
# ============================================================
# CELL 1 — SETUP
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from itertools import combinations
import math
import numpy as np
import pandas as pd

from scipy.stats import binomtest, norm

PROJECT = Path('/content/drive/MyDrive/Cataract')

LOCKED = (
    PROJECT
    / 'FINAL_REVISION_2026_08'
    / 'locked_final_benchmark'
)

OUT = (
    PROJECT
    / 'FINAL_REVISION_2026_08'
    / 'final_statistical_tests'
)

OUT.mkdir(parents=True, exist_ok=True)

RUNS = {
    'MobileNetV2':
        PROJECT
        / 'FINAL_REVISION_2026_08'
        / 'reviewer_10_2_clean_split'
        / 'MobileNetV2_frozen',

    'DenseNet201':
        PROJECT
        / 'FINAL_REVISION_2026_08'
        / 'reviewer_10_2_clean_split'
        / 'DenseNet201_frozen',

    'InceptionResNetV2':
        PROJECT
        / 'FINAL_REVISION_2026_08'
        / 'clean_split_models'
        / 'InceptionResNetV2_CORRECTED_13Layer_Frozen',

    'ResNet152V2':
        PROJECT
        / 'FINAL_REVISION_2026_08'
        / 'clean_split_models'
        / 'ResNet152V2',

    'Xception':
        PROJECT
        / 'FINAL_REVISION_2026_08'
        / 'clean_split_models'
        / 'Xception'
}

MODELS = list(RUNS.keys())
ALPHA = 0.05

assert LOCKED.exists(), f'STOP: Missing locked benchmark folder: {LOCKED}'

print('Locked benchmark:', LOCKED)
print('Statistical-test output:', OUT)
print('Models:', MODELS)
print('\n✅ CELL 1 COMPLETE')

In [ ]:
# ============================================================
# CELL 2 — LOAD + VERIFY FINAL PREDICTION ARRAYS
# ============================================================

loaded = {}

reference_y = None
reference_paths = None

for model, folder in RUNS.items():

    probs = np.load(folder / 'probs.npy')
    y_true = np.load(folder / 'y_true.npy')
    y_pred = np.load(folder / 'y_pred.npy')
    idx = pd.read_csv(folder / 'test_predictions_index.csv')

    assert probs.shape == (2587, 3)
    assert y_true.shape == (2587,)
    assert y_pred.shape == (2587,)
    assert len(idx) == 2587
    assert np.array_equal(y_pred, probs.argmax(axis=1))
    assert np.array_equal(y_true, idx['y_true'].to_numpy())
    assert np.array_equal(y_pred, idx['y_pred'].to_numpy())

    paths = idx['filepath'].astype(str).to_numpy()

    if reference_y is None:
        reference_y = y_true.copy()
        reference_paths = paths.copy()
    else:
        assert np.array_equal(reference_y, y_true), (
            f'STOP: y_true mismatch for {model}'
        )
        assert np.array_equal(reference_paths, paths), (
            f'STOP: filepath ordering mismatch for {model}'
        )

    loaded[model] = {
        'probs': probs,
        'y_true': y_true,
        'y_pred': y_pred
    }

print('Cataract:', int((reference_y == 0).sum()))
print('Normal:', int((reference_y == 1).sum()))
print('Not Eye:', int((reference_y == 2).sum()))
print('Total:', len(reference_y))

assert int((reference_y == 0).sum()) == 854
assert int((reference_y == 1).sum()) == 977
assert int((reference_y == 2).sum()) == 756

print('\n✅ ALL FIVE FINAL ARRAYS ALIGNED')
print('✅ CELL 2 COMPLETE')

In [ ]:
# ============================================================
# CELL 3 — HOLM–BONFERRONI CORRECTION
# ============================================================

def holm_adjust(p_values):

    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)

    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.empty(m, dtype=float)

    running_max = 0.0

    for i, p in enumerate(sorted_p):
        adj = (m - i) * p
        running_max = max(running_max, adj)
        adjusted_sorted[i] = min(running_max, 1.0)

    adjusted = np.empty(m, dtype=float)
    adjusted[order] = adjusted_sorted

    return adjusted


# Quick sanity check
_test = holm_adjust([0.001, 0.01, 0.04])
assert np.all((_test >= 0) & (_test <= 1))

print('✅ Holm–Bonferroni function ready')
print('✅ CELL 3 COMPLETE')

In [ ]:
# ============================================================
# CELL 4 — EXACT McNEMAR TESTS ON 3-CLASS CORRECTNESS
# ============================================================

mcnemar_rows = []

for model_a, model_b in combinations(MODELS, 2):

    y_true = reference_y

    pred_a = loaded[model_a]['y_pred']
    pred_b = loaded[model_b]['y_pred']

    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)

    # b = A correct / B wrong
    b = int(np.sum(correct_a & ~correct_b))

    # c = A wrong / B correct
    c = int(np.sum(~correct_a & correct_b))

    discordant = b + c

    if discordant == 0:
        p_raw = 1.0
    else:
        # Exact two-sided McNemar test:
        # under H0, either model wins each discordant pair with p=0.5.
        p_raw = float(
            binomtest(
                min(b, c),
                n=discordant,
                p=0.5,
                alternative='two-sided'
            ).pvalue
        )

    acc_a = float(correct_a.mean())
    acc_b = float(correct_b.mean())

    mcnemar_rows.append({
        'Model_A': model_a,
        'Model_B': model_b,
        'Accuracy_A': acc_a,
        'Accuracy_B': acc_b,
        'Accuracy_Difference_pp': 100 * (acc_a - acc_b),
        'A_correct_B_wrong': b,
        'A_wrong_B_correct': c,
        'Discordant_Pairs': discordant,
        'McNemar_p_raw': p_raw
    })

mcnemar_df = pd.DataFrame(mcnemar_rows)

mcnemar_df['McNemar_p_Holm'] = holm_adjust(
    mcnemar_df['McNemar_p_raw'].to_numpy()
)

mcnemar_df['Significant_After_Holm'] = (
    mcnemar_df['McNemar_p_Holm'] < ALPHA
)

mcnemar_df.to_csv(
    OUT / 'FINAL_McNemar_Holm.csv',
    index=False
)

print('========================================')
print('FINAL McNEMAR + HOLM RESULTS')
print('========================================')

display(
    mcnemar_df.round(6)
)

print('\nSignificant after Holm:')
display(
    mcnemar_df[
        mcnemar_df['Significant_After_Holm']
    ][[
        'Model_A',
        'Model_B',
        'Accuracy_Difference_pp',
        'McNemar_p_raw',
        'McNemar_p_Holm'
    ]].round(6)
)

print('\n✅ CELL 4 COMPLETE')

## DeLong implementation

For the clinical ROC comparison:

- only true Cataract and Normal test images are used;
- Cataract is the positive class;
- the clinical score is  
  `P(Cataract) / [P(Cataract) + P(Normal)]`;
- DeLong compares the correlated ROC AUC estimates because all models are evaluated on the same clinical images.

The implementation below follows the standard fast DeLong covariance formulation.

In [ ]:
# ============================================================
# CELL 5 — FAST DeLong FUNCTIONS
# ============================================================

def compute_midrank(x):

    x = np.asarray(x)
    order = np.argsort(x)
    z = x[order]

    n = len(x)
    ranks = np.empty(n, dtype=float)

    i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1

        # 1-based average rank
        avg_rank = 0.5 * (i + j - 1) + 1.0
        ranks[i:j] = avg_rank
        i = j

    out = np.empty(n, dtype=float)
    out[order] = ranks

    return out


def fast_delong(predictions_sorted_transposed, label_1_count):

    m = int(label_1_count)
    n = predictions_sorted_transposed.shape[1] - m
    k = predictions_sorted_transposed.shape[0]

    positive = predictions_sorted_transposed[:, :m]
    negative = predictions_sorted_transposed[:, m:]

    tx = np.empty((k, m), dtype=float)
    ty = np.empty((k, n), dtype=float)
    tz = np.empty((k, m + n), dtype=float)

    for r in range(k):
        tx[r, :] = compute_midrank(positive[r, :])
        ty[r, :] = compute_midrank(negative[r, :])
        tz[r, :] = compute_midrank(
            predictions_sorted_transposed[r, :]
        )

    aucs = (
        tz[:, :m].sum(axis=1) / m / n
        - (m + 1.0) / (2.0 * n)
    )

    v01 = (
        tz[:, :m] - tx
    ) / n

    v10 = (
        1.0
        - (
            tz[:, m:] - ty
        ) / m
    )

    sx = np.atleast_2d(np.cov(v01))
    sy = np.atleast_2d(np.cov(v10))

    delong_cov = sx / m + sy / n

    return aucs, delong_cov


def delong_pair_test(y_true_binary, score_a, score_b):

    y_true_binary = np.asarray(
        y_true_binary,
        dtype=int
    )

    score_a = np.asarray(
        score_a,
        dtype=float
    )

    score_b = np.asarray(
        score_b,
        dtype=float
    )

    assert set(np.unique(y_true_binary)) == {0, 1}

    # Positives first, negatives second.
    order = np.argsort(
        -y_true_binary
    )

    label_1_count = int(
        y_true_binary.sum()
    )

    preds = np.vstack([
        score_a,
        score_b
    ])[:, order]

    aucs, cov = fast_delong(
        preds,
        label_1_count
    )

    contrast = np.array([1.0, -1.0])

    variance = float(
        contrast @ cov @ contrast.T
    )

    diff = float(
        aucs[0] - aucs[1]
    )

    if variance <= 0:
        if abs(diff) < 1e-15:
            z = 0.0
            p = 1.0
        else:
            z = np.inf if diff > 0 else -np.inf
            p = 0.0
    else:
        z = diff / math.sqrt(variance)
        p = 2.0 * norm.sf(abs(z))

    return {
        'AUC_A': float(aucs[0]),
        'AUC_B': float(aucs[1]),
        'AUC_Difference': diff,
        'z': float(z),
        'p_raw': float(p)
    }


print('✅ Fast DeLong functions ready')
print('✅ CELL 5 COMPLETE')

In [ ]:
# ============================================================
# CELL 6 — BUILD FINAL CLINICAL SCORES
# ============================================================

clinical_mask = np.isin(
    reference_y,
    [0, 1]
)

clinical_true3 = reference_y[
    clinical_mask
]

y_binary = (
    clinical_true3 == 0
).astype(int)

clinical_scores = {}

for model in MODELS:

    probs = loaded[model]['probs'][
        clinical_mask
    ]

    denominator = (
        probs[:, 0]
        + probs[:, 1]
    )

    score = np.divide(
        probs[:, 0],
        denominator,
        out=np.full(
            denominator.shape,
            0.5,
            dtype=float
        ),
        where=denominator > 0
    )

    clinical_scores[model] = score

print('Clinical N:', len(y_binary))
print('Cataract positives:', int(y_binary.sum()))
print('Normal negatives:', int((y_binary == 0).sum()))

assert len(y_binary) == 1831
assert int(y_binary.sum()) == 854
assert int((y_binary == 0).sum()) == 977

print('\n✅ CLINICAL SCORES READY')
print('✅ CELL 6 COMPLETE')

In [ ]:
# ============================================================
# CELL 7 — PAIRWISE DeLong TESTS + HOLM CORRECTION
# ============================================================

delong_rows = []

for model_a, model_b in combinations(MODELS, 2):

    result = delong_pair_test(
        y_binary,
        clinical_scores[model_a],
        clinical_scores[model_b]
    )

    delong_rows.append({
        'Model_A': model_a,
        'Model_B': model_b,
        'AUC_A': result['AUC_A'],
        'AUC_B': result['AUC_B'],
        'AUC_Difference': result['AUC_Difference'],
        'DeLong_z': result['z'],
        'DeLong_p_raw': result['p_raw']
    })

delong_df = pd.DataFrame(
    delong_rows
)

delong_df['DeLong_p_Holm'] = holm_adjust(
    delong_df['DeLong_p_raw'].to_numpy()
)

delong_df['Significant_After_Holm'] = (
    delong_df['DeLong_p_Holm'] < ALPHA
)

delong_df.to_csv(
    OUT / 'FINAL_DeLong_Holm.csv',
    index=False
)

print('========================================')
print('FINAL DeLong + HOLM RESULTS')
print('========================================')

display(
    delong_df.round(8)
)

print('\nSignificant after Holm:')
display(
    delong_df[
        delong_df['Significant_After_Holm']
    ][[
        'Model_A',
        'Model_B',
        'AUC_Difference',
        'DeLong_p_raw',
        'DeLong_p_Holm'
    ]].round(8)
)

print('\n✅ CELL 7 COMPLETE')

In [ ]:
# ============================================================
# CELL 8 — MERGED MANUSCRIPT-READY PAIRWISE TABLE
# ============================================================

merged = mcnemar_df.merge(
    delong_df,
    on=['Model_A', 'Model_B'],
    how='inner',
    suffixes=('_McNemar', '_DeLong')
)

final_table = merged[[
    'Model_A',
    'Model_B',
    'Accuracy_Difference_pp',
    'A_correct_B_wrong',
    'A_wrong_B_correct',
    'McNemar_p_raw',
    'McNemar_p_Holm',
    'Significant_After_Holm_McNemar',
    'AUC_A',
    'AUC_B',
    'AUC_Difference',
    'DeLong_p_raw',
    'DeLong_p_Holm',
    'Significant_After_Holm_DeLong'
]].copy()

final_table.to_csv(
    OUT / 'MANUSCRIPT_FINAL_Pairwise_Statistical_Tests.csv',
    index=False
)

print('========================================')
print('MANUSCRIPT-READY FINAL PAIRWISE TABLE')
print('========================================')

display(
    final_table.round(8)
)

print('\n✅ CELL 8 COMPLETE')

In [ ]:
# ============================================================
# CELL 9 — AUTOMATIC INTERPRETATION SUMMARY
# ============================================================

sig_mcnemar = mcnemar_df[
    mcnemar_df['Significant_After_Holm']
].copy()

sig_delong = delong_df[
    delong_df['Significant_After_Holm']
].copy()

lines = []

lines.append(
    f'Exact pairwise McNemar tests were performed on the '
    f'2,587-image three-class test set, with Holm–Bonferroni '
    f'correction across {len(mcnemar_df)} pairwise comparisons.'
)

lines.append(
    f'Pairwise DeLong tests were performed on the '
    f'1,831-image Cataract-v-Normal clinical subset, with '
    f'Holm–Bonferroni correction across {len(delong_df)} '
    f'pairwise comparisons.'
)

lines.append(
    f'After Holm correction, {len(sig_mcnemar)} of '
    f'{len(mcnemar_df)} McNemar comparisons remained significant.'
)

lines.append(
    f'After Holm correction, {len(sig_delong)} of '
    f'{len(delong_df)} DeLong comparisons remained significant.'
)

summary_text = '\n'.join(lines)

(
    OUT
    / 'FINAL_Statistical_Test_Summary.txt'
).write_text(summary_text)

print('========================================')
print('STATISTICAL-TEST SUMMARY')
print('========================================')
print(summary_text)

if len(sig_mcnemar):
    print('\nSignificant McNemar pairs:')
    for _, r in sig_mcnemar.iterrows():
        print(
            f"- {r['Model_A']} vs {r['Model_B']}: "
            f"Holm p={r['McNemar_p_Holm']:.6g}"
        )
else:
    print('\nNo McNemar pair remained significant after Holm correction.')

if len(sig_delong):
    print('\nSignificant DeLong pairs:')
    for _, r in sig_delong.iterrows():
        print(
            f"- {r['Model_A']} vs {r['Model_B']}: "
            f"Holm p={r['DeLong_p_Holm']:.6g}"
        )
else:
    print('\nNo DeLong pair remained significant after Holm correction.')

print('\n✅ CELL 9 COMPLETE')

In [ ]:
# ============================================================
# CELL 10 — FINAL COMPLETION CHECK
# ============================================================

required = [
    'FINAL_McNemar_Holm.csv',
    'FINAL_DeLong_Holm.csv',
    'MANUSCRIPT_FINAL_Pairwise_Statistical_Tests.csv',
    'FINAL_Statistical_Test_Summary.txt'
]

missing = [
    fn
    for fn in required
    if not (OUT / fn).exists()
]

if missing:

    print('Missing:')
    for fn in missing:
        print('❌', fn)

    raise RuntimeError(
        'STOP: Statistical testing is incomplete.'
    )

(
    OUT
    / 'STATISTICAL_TESTS_DONE.txt'
).write_text(
    'Final pairwise McNemar and DeLong tests completed.\n'
    'Holm–Bonferroni correction applied separately within each test family.\n'
    'McNemar: exact two-sided binomial version.\n'
    'DeLong: correlated ROC AUC comparison on clinical Cataract-v-Normal subset.\n'
    'Alpha: 0.05\n'
)

print('========================================')
print('✅ FINAL STATISTICAL TESTS COMPLETE')
print('========================================')

print('\nSaved to:')
print(OUT)

print(
    '\nNEXT: Upload this executed notebook to ChatGPT.'
)

print(
    '\nNO MODEL TRAINING IS REQUIRED FOR THIS STEP.'
)